In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import mygene
from pathlib import Path
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Inital setting for plot size
from matplotlib import rcParams
FIGSIZE = (4, 4)
rcParams["figure.figsize"] = FIGSIZE

from icepop.data import HomologyData
from icepop.convert_score import CrossSpeciesScoreConverter

In [2]:
adata = sc.read('../data/mouse_ileum/mouse_ileum_normed.h5ad')
adata.obs['metacell'] = pd.read_csv('../results/mouse_ileum_mc/mc_assign.csv', header=None, index_col=None)[0].values

In [3]:
# convert mouse to human genes
ortho_map = HomologyData(sp='mmusculus').load()
m2h = {}
for h, m in ortho_map.items():
    for i in m:
        m2h[i] = h
m2h['116903'] = '797'

adata.var['hentrez'] = [m2h[i] if i in m2h else None for i in adata.var_names]
entrezid = adata.var['hentrez'][~pd.isna(adata.var['hentrez'])].values

mg = mygene.MyGeneInfo()
res = mg.querymany(
    entrezid,
    scopes="entrezgene",
    fields="symbol",
    species="human"
)

entrez_to_symbol = {
    r["query"]: r.get("symbol", None)
    for r in res
}
adata.var['hsymbol'] = [entrez_to_symbol[i] if i in entrez_to_symbol else None for i in adata.var['hentrez']]

adata = adata[:, ~pd.isna(adata.var['hsymbol'])].copy()
adata.var_names = adata.var['hsymbol']

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


In [4]:
indir = '../results/mouse_ileum_icepop'

trait = 'asd'
mc_asso = f'{indir}/metacell__trait-{trait}.csv'
ct_asso = f'{indir}/celltype__trait-{trait}.csv'

mc_df = pd.read_csv(mc_asso, header=0, index_col=None)
mc_df = mc_df.set_index('metacell')
ct_df = pd.read_csv(ct_asso, header=0, index_col=None)
ct_df = ct_df.set_index('cell_type')

# add mc count in each ct
min_purity = 0.2
freq_df = pd.crosstab(adata.obs['cell_type'], adata.obs['metacell'])
freq_df = freq_df.div(freq_df.sum(0))
ct2meta = {ct: np.asarray(row[row >= min_purity].index) for ct, row in freq_df.iterrows()}
ct_df['n_mc'] = [len(ct2meta[ct]) for ct in ct_df.index]

In [5]:
ct_df['discovery'] = ct_df['q'] <= 0.1
associated_cts = ct_df[ct_df['discovery']].index.values
ct_df[ct_df['discovery']]

,beta,se,z,p,q,sig_pct,n_mc,discovery
cell_type,,,,,,,,
Neuron,4.226135,1.193132,3.542053,0.002393,0.028713,0.214493,16,True
Inhibitory_motor_neuron,1.888837,0.685965,2.753545,0.002183,0.028713,0.748031,2,True


## check influ genes in SFARI genes

In [6]:
sf_df = pd.read_csv('../data/mouse_colon/SFARI-Gene_genes_01-13-2025release_03-27-2025export.csv', header=0, index_col=None)
mg = mygene.MyGeneInfo()
results = mg.querymany(sf_df['ensembl-id'], scopes='ensemblgene', fields='entrezgene', species='human')
ensembl2entrez = {r['query']: r.get('entrezgene', None) for r in results}
sf_df['entrez'] = [ensembl2entrez[ens] if ens in ensembl2entrez else None for ens in sf_df['ensembl-id']]
sf_df = sf_df[~pd.isna(sf_df['entrez'])].copy()

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
11 input query terms found no hit:	['nan', 'nan', 'nan', 'nan', 'nan', 'nan', 'nan', 'nan', 'nan', 'nan', 'nan']


In [7]:
f = np.load('../results/mouse_ileum_icepop/dfbs__trait-asd.npz', allow_pickle=True)
dfbs = pd.DataFrame(f['dfbs'], index=f['celltypes'], columns=f['genes'])

## check top influential genes in each cell groups

In [8]:
e2sym = dict(zip(adata.var['hentrez'], adata.var['hsymbol']))
shared_entrez = dfbs.columns.intersection(adata.var['hentrez'].values)
shared_dfbs = dfbs.loc[:, shared_entrez].copy()

In [9]:
top_dfbs_df = []
for idx, row in shared_dfbs.iterrows():
    row = row.sort_values(ascending=False)
    row = row[row >= (2 / np.sqrt(dfbs.shape[1]))]
    row = pd.DataFrame(row)
    row = row.reset_index()
    row.columns = ['Entrez', 'Influence score']
    row['Influence score rank'] = row['Influence score'].rank(ascending=False)
    row['Cell type'] = idx
    row['FDR'] = ct_df.loc[idx, 'q']
    row['Associated'] = ct_df.loc[idx, 'discovery']
    top_dfbs_df.append(row)
top_dfbs_df = pd.concat(top_dfbs_df, axis=0, ignore_index=True)
top_dfbs_df['Gene'] = [e2sym[i] for i in top_dfbs_df['Entrez']]
top_dfbs_df['In SFARI'] = top_dfbs_df['Entrez'].isin(sf_df['entrez'])
top_dfbs_df = top_dfbs_df.loc[:, ['Gene', 'Entrez', 'Cell type', 'FDR', 'Associated', 'Influence score', 'Influence score rank', 'In SFARI']].copy()
top_dfbs_df.to_csv('../results/ASD_GI/influ_genes_all_ileum.tsv', header=True, index=False, sep='\t')

In [10]:
sf_idx_df = sf_df.set_index('entrez')
top_sf_df = top_dfbs_df[top_dfbs_df['In SFARI']].copy()
top_sf_df = top_sf_df.reset_index(drop=True)

top_sf_df = pd.concat(
    [
        top_sf_df,
        sf_idx_df.loc[top_sf_df['Entrez'], :].reset_index(drop=False)
    ],
    axis=1
)
top_sf_df = top_sf_df.drop(columns=['entrez', 'status', 'gene-symbol', 'In SFARI'])
new_order = [
    "Gene", "gene-name", "Entrez", "ensembl-id", "chromosome",
    "Cell type", "FDR", "Associated", "Influence score", 'Influence score rank',
    "genetic-category", "gene-score", "eagle", "syndromic", "number-of-reports"
]
top_sf_df = top_sf_df[new_order]
top_sf_df.to_csv('../results/ASD_GI/influ_genes_SFARI_ileum.tsv', header=True, index=False, sep='\t')